In [16]:
total_cyclists = cyclist_profile['FE_PESS'].sum()
total_pnb_cyclists = pnb_cyclist_profile['FE_PESS'].sum()

zone_profile = {}

for zona in cyclist_profile['ZONA'].unique():
    zone_profile[zona] = {
        'total_cyclists': cyclist_profile[cyclist_profile['ZONA'] == zona]['FE_PESS'].sum(),
        'total_pnb_cyclists': pnb_cyclist_profile[pnb_cyclist_profile['ZONA'] == zona]['FE_PESS'].sum(),
        'sexo':
    }
     

,FE_PESS,ZONA,SEXO,IDADE,RAÇA,RENDA_FA,GRAU_INS,TIPO_DOM,DURACAO,DISTANCIA,PE_BICI,MOT_SRES,QT_AUTO,QT_MOTO
0,20.452542,1,1,67,4,7000.000000,4,1,30,1518.600013,7,2,1,0
93,20.805172,1,1,28,4,5779.081275,2,1,8,627.498207,1,3,1,0
350,36.085220,2,2,34,1,9433.288101,5,1,10,928.282823,7,3,1,0
354,89.778288,2,1,36,4,9433.288101,5,1,10,928.282823,7,3,1,0
371,222.072926,3,1,31,4,3000.000000,4,1,15,2220.216656,6,3,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75637,41.867395,341,2,48,1,8795.540000,5,1,15,878.080862,1,3,0,0
75639,19.614564,341,1,74,1,8795.540000,5,1,15,815.726670,7,7,0,0
75641,19.614564,341,2,73,1,8795.540000,5,1,15,815.726670,7,7,0,0
75836,166.642369,342,1,49,1,10000.000000,5,1,30,6261.804373,6,3,0,0


# PROBLEMAS

In [74]:
from shapely.validation import explain_validity
invalid_OD_zones_gdf = OD_zones_gdf_untreated[~OD_zones_gdf_untreated.is_valid]
explain_validity(invalid_OD_zones_gdf)

# Por muitos polígonos não são válidos, (declarado na célula 3) segue a lista de razões:

,geometry
8,Ring Self-intersection[334100.113609622 739806...
10,Self-intersection[335766.610853515 7397060.310...
11,Ring Self-intersection[335762.685647727 739705...
39,Ring Self-intersection[338658.671142381 739572...
40,Ring Self-intersection[338658.704127304 739572...
...,...
511,Ring Self-intersection[315652.949310819 740127...
512,Ring Self-intersection[304370.332846271 739657...
513,Ring Self-intersection[304370.332846271 739657...
517,Self-intersection[304438.957977715 7382590.458...


# CEMITÉRIO

In [27]:
import pandas as pd

def read_ibge_microdata(txt_path, layout_path, sheet_name='PESS'):
    # Read layout sheet
    layout = pd.read_excel(layout_path, sheet_name=sheet_name, engine='odf').iloc[1:]
    
    # Extract variable names and positions
    var_names = layout.iloc[:, 0].astype(str).tolist()
    starts = layout.iloc[:, 7].astype(float).astype(int).tolist()
    ends = layout.iloc[:, 8].astype(float).astype(int).tolist()
    
    # Build colspecs
    colspecs = [(start - 1, end) for start, end in zip(starts, ends)]
    
    # Read fixed-width file
    df = pd.read_fwf(txt_path, colspecs=colspecs, names=var_names)
    
    return df

txt_path = '.\\data\\Amostra_Pessoas_14munic.txt'
layout_path = '.\\Layout_microdados_Amostra.ods'

df = read_ibge_microdata(txt_path, layout_path)


In [46]:
dict_code_prof = {
    'V0601':{'sexo':{'1':'Masculino','2':'Feminino'}},
    'V0606':{'cor':{'1':'Branca','2':'Preta','3':'Amarela','4':'Parda','5':'Indigena','9':'Sem_declaracao'}},
    'V6400':{'nivel_de_instrucao':{'1':'Sem_instrucao','2':'fundamental_completo','3':'medio_completo','4':'superior_completo'}},
    'V0640':{'estado_civil':{'1':'casado','5':'solteiro','4':'viuvo','3':'divorciado','2':'desquitado'}},
    'V0645':{'trabalho':{'1':'um','2':'dois_ou_mais','':'nenhum'}},
    'V6526':'renda_em_salarios_minimos',
    'V0653':'horas_trabalhadas',
    'V0662':{'tempo_de_deslocamento_ao_trabalho':{'1':'<=5min','2':'6-30min','3':'31min-1h','4':'1-2h','5':'>2h'}},
    'V6940':{'categoria_profissional':{'1':'empregado_clt','2':'empregado_estatuario(militares_inclusos)','3':'empregado_sem_clt','4':'conta_propria','5':'empregador','6':'nao_remunerado','7':'trabalhador_subsistente'}},
    'V5080':'rendimento_familiar_per_capita_em_salarios_minimos',
    'V1005':{'situacao_do_setor':{'1':'area_urbanizada','2':'area_nao_urbanizada','3':'area_urbanizada_isolada','4':'area_rural_de_extensao_urbana','5':'aglomerado_rural','6':'aglomerado_rural','7':'aglomerado_rural','8':'area_rural_exclusive_aglomerado_rural'}},
    'V0221':{'motocicleta_para_uso_particular':{'1':'possui','2':'nao_possui'}},
    'V0222':{'automovel_para_uso_particular':{'1':'possui','2':'nao_possui'}},
}

# Separando os ciclistas gerais dos ciclistas vivendo na zona PNB

## eu preciso associar as linhas de pessoas com os setores do censo para perfilar cada zona - agora com os microdados. MAS os microdados estão desencontrados, os códigos de região não batem, preciso resolver isso.

In [65]:
a = census_microdata['setor'].unique() 
b = sp_zones_dissolved['setor'].unique()
c = [item for item in a if item not in b]
print(len(a), len(b), len(c))

310 240 310


In [14]:
import geopandas as gpd

sp_zones = gpd.read_file('./data/sao_paulo_demographics.geojson')
sp_zones['setor_cens'] = sp_zones['setor_cens'].str[:-2]
sp_zones_dissolved = sp_zones.dissolve(by='setor_cens')
sp_zones_dissolved = sp_zones_dissolved.reset_index()
sp_zones_dissolved = sp_zones_dissolved.rename(columns={'setor_cens': 'setor'})
census_microdata = gpd.read_file('./data/biker_profile.csv')
census_microdata = census_microdata.merge(sp_zones_dissolved[['setor', 'geometry']], on='setor', how='left')



In [70]:
import geopandas as gpd
from shapely import union_all 
from shapely import make_valid 

OD_data_df = gpd.read_file('./data/od23_all.csv')

OD_data_gdf = gpd.GeoDataFrame(
    OD_data_df, 
    geometry=gpd.points_from_xy(OD_data_df['CO_DOM_Y'], OD_data_df['CO_DOM_X']),
    crs="EPSG:4326"
)
OD_data_gdf = OD_data_gdf.to_crs(epsg=31983)

OD_zones_gdf_untreated = gpd.read_file('./data/Zonas_2023.shp')
OD_zones_gdf = make_valid(OD_zones_gdf_untreated)
OD_zones_gdf.set_crs(epsg=31983, inplace=True)



spsp_limits = gpd.read_file('./data/REGIAO5/SIRGAS_REGIAO5.shp')

OD_zones_spsp = OD_zones_gdf.clip(spsp_limits.union_all())

spsp_data = OD_data_gdf[OD_data_gdf.geometry.within(OD_zones_spsp.union_all())]

spsp_cyclist_data = spsp_data['MODOPRIN'] == '16'

spsp_OD_data_gdf = spsp_data[spsp_cyclist_data]
spsp_OD_data_gdf = spsp_OD_data_gdf.drop(index=spsp_OD_data_gdf.index[1::2])
# this dataframe contain the data of all cyclists living in Sao Paulo city
# in this particular dataset, those are all cyclists.

pnb_zone_gdf = gpd.read_file('./data/pnb_zone.shp')
pnb_zone = pnb_zone_gdf['geometry'][0]

spsp_pnb_OD = spsp_OD_data_gdf.geometry.apply(lambda p: pnb_zone.contains(p))
# long nonsense abreviation, sorry. It means the cyclists living in Sao Paulo's pnb zone

spsp_pnb_OD_data_gdf = spsp_OD_data_gdf[spsp_pnb_OD]
# this dataframe contains the cyclists living in the PNB zone of Sao Paulo city


### Teste: Há 'ciclistas secundários'?

In [3]:
secondary_cyclists = spsp_pnb_OD_data_gdf[(spsp_pnb_OD_data_gdf['MODOPRIN'].astype('int') != '16') &
                                          ((spsp_pnb_OD_data_gdf['MODO1'] == '16') |
                                           (spsp_pnb_OD_data_gdf['MODO2'] == '16') |
                                           (spsp_pnb_OD_data_gdf['MODO3'] == '16') |
                                           (spsp_pnb_OD_data_gdf['MODO4'] == '16')                                           
                                           )]
secondary_cyclists[['MODOPRIN', 'MODO1', 'MODO2', 'MODO3', 'MODO4']]
# why is row zero present? 

,MODOPRIN,MODO1,MODO2,MODO3,MODO4
0,16,16,0,0,0
1,16,16,0,0,0
93,16,16,0,0,0
94,16,16,0,0,0
371,16,16,0,0,0
...,...,...,...,...,...
75641,16,16,0,0,0
75835,16,16,0,0,0
75836,16,16,0,0,0
75858,16,16,0,0,0


## Em geral, não

In [134]:
cyclist_profile = spsp_OD_data_gdf[['FE_PESS', 
                                    'ZONA',
                                    'SEXO', 
                                    'IDADE', 
                                    'RAÇA', 
                                    'RENDA_FA', 
                                    'GRAU_INS', 
                                    'TIPO_DOM',
                                    'DURACAO',
                                    'DISTANCIA',
                                    'PE_BICI',
                                    'MOT_SRES',
                                    'QT_AUTO',
                                    'QT_MOTO'
                                    ]]

cyclist_profile = cyclist_profile.astype({
    'FE_PESS': 'float',
    'RENDA_FA': 'float',
    'DURACAO': 'int',
    'DISTANCIA': 'float',
    'QT_AUTO': 'int',
    'QT_MOTO': 'int'})

pnb_cyclist_profile = spsp_pnb_OD_data_gdf[['FE_PESS',
                                            'ZONA',
                                            'SEXO', 
                                            'IDADE', 
                                            'RAÇA', 
                                            'RENDA_FA', 
                                            'GRAU_INS', 
                                            'TIPO_DOM',
                                            'DURACAO',
                                            'DISTANCIA',
                                            'PE_BICI',
                                            'MOT_SRES',
                                            'QT_AUTO',
                                            'QT_MOTO'
                                            ]]

pnb_cyclist_profile = pnb_cyclist_profile.astype({
    'FE_PESS': 'float',
    'IDADE': 'int',
    'RENDA_FA': 'float',
    'DURACAO': 'int',
    'DISTANCIA': 'float',
    'QT_AUTO': 'int',
    'QT_MOTO': 'int'})
    
dict_code_prof = {
    'SEXO':{'1':'Masculino','2':'Feminino', '3':'Nao_Respondeu'},
    'RAÇA':{'1':'Branca','2':'Preta','3':'Amarela','4':'Parda','5':'Indigena','6':'Sem_declaracao'},
    'GRAU_INS':{'1':'Sem_instrucao','2':'fundamental1_completo','3':'fundamental2_completo','4':'medio_completo','5':'superior_completo'},
    'TIPO_DOM':{'1':'particular','2':'coletivo'},
    'MOT_SRES':{'1':'Trabalho_Industria','2':'Trabalho_Comercio','3':'Trabalho_Servicos','4':'Escola_Educacao','5':'Compras','6':'Saude','7':'Lazer','8':'Residencia','9':'Busca_Emprego','10':'Assuntos_Pessoais','11':'Refeicoes'},
}

for col, mapping in dict_code_prof.items():
    if isinstance(mapping, dict):
        cyclist_profile[col] = cyclist_profile[col].map(mapping)
        pnb_cyclist_profile[col] = pnb_cyclist_profile[col].map(mapping)

In [170]:
cyclist_profile

,SEXO,PE_BICI,TIPO_DOM,GRAU_INS,RAÇA,MOT_SRES,QT_MOTO,IDADE,RENDA_FA,DISTANCIA,QT_AUTO,DURACAO
ZONA,,,,,,,,,,,,
1,"SEXO Masculino 1.0 Name: FE_PESS, dtype: fl...",PE_BICI 1 0.504274 7 0.495726 Name: FE_P...,"TIPO_DOM particular 1.0 Name: FE_PESS, dtyp...",GRAU_INS fundamental1_completo 0.504274 med...,"RAÇA Parda 1.0 Name: FE_PESS, dtype: float64",MOT_SRES Trabalho_Comercio 0.495726 Trabalh...,0.000000,47.333333,6384.323036,1069.240983,1.000000,18.905983
10,"SEXO Masculino 1.0 Name: FE_PESS, dtype: fl...",PE_BICI 1 0.877839 4 0.122161 Name: FE_P...,"TIPO_DOM particular 1.0 Name: FE_PESS, dtyp...",GRAU_INS fundamental2_completo 0.122161 med...,RAÇA Branca 0.729960 Parda 0.122161 Pre...,MOT_SRES Escola_Educacao 0.729960 Trabalh...,0.000000,28.736389,2324.926272,1177.690497,0.000000,14.723530
100,"SEXO Masculino 1.0 Name: FE_PESS, dtype: fl...","PE_BICI 7 1.0 Name: FE_PESS, dtype: float64","TIPO_DOM particular 1.0 Name: FE_PESS, dtyp...",GRAU_INS superior_completo 1.0 Name: FE_PES...,"RAÇA Branca 1.0 Name: FE_PESS, dtype: float64",MOT_SRES Trabalho_Servicos 1.0 Name: FE_PES...,0.000000,61.000000,24000.000000,5535.948880,1.000000,35.000000
102,"SEXO Feminino 1.0 Name: FE_PESS, dtype: flo...","PE_BICI 7 1.0 Name: FE_PESS, dtype: float64","TIPO_DOM particular 1.0 Name: FE_PESS, dtyp...","GRAU_INS medio_completo 1.0 Name: FE_PESS, ...","RAÇA Branca 1.0 Name: FE_PESS, dtype: float64",MOT_SRES Trabalho_Servicos 1.0 Name: FE_PES...,0.000000,21.000000,5024.091899,7683.715573,1.000000,40.000000
103,SEXO Feminino 0.726236 Masculino 0.2737...,PE_BICI 1 0.273764 7 0.726236 Name: FE_P...,"TIPO_DOM particular 1.0 Name: FE_PESS, dtyp...",GRAU_INS superior_completo 1.0 Name: FE_PES...,"RAÇA Branca 1.0 Name: FE_PESS, dtype: float64","MOT_SRES Lazer 1.0 Name: FE_PESS, dtype: fl...",0.000000,33.441065,8945.969582,5850.010166,1.273764,25.893536
...,...,...,...,...,...,...,...,...,...,...,...,...
94,"SEXO Masculino 1.0 Name: FE_PESS, dtype: fl...","PE_BICI 7 1.0 Name: FE_PESS, dtype: float64","TIPO_DOM particular 1.0 Name: FE_PESS, dtyp...",GRAU_INS superior_completo 1.0 Name: FE_PES...,"RAÇA Branca 1.0 Name: FE_PESS, dtype: float64",MOT_SRES Trabalho_Servicos 1.0 Name: FE_PES...,0.000000,22.000000,16514.598066,5781.864838,1.000000,30.000000
95,"SEXO Masculino 1.0 Name: FE_PESS, dtype: fl...","PE_BICI 1 1.0 Name: FE_PESS, dtype: float64","TIPO_DOM particular 1.0 Name: FE_PESS, dtyp...",GRAU_INS superior_completo 1.0 Name: FE_PES...,"RAÇA Branca 1.0 Name: FE_PESS, dtype: float64",MOT_SRES Trabalho_Servicos 1.0 Name: FE_PES...,0.000000,60.000000,13386.473021,2531.063018,1.000000,15.000000
96,"SEXO Masculino 1.0 Name: FE_PESS, dtype: fl...","PE_BICI 1 1.0 Name: FE_PESS, dtype: float64","TIPO_DOM particular 1.0 Name: FE_PESS, dtyp...",GRAU_INS superior_completo 1.0 Name: FE_PES...,"RAÇA Amarela 1.0 Name: FE_PESS, dtype: float64",MOT_SRES Trabalho_Servicos 1.0 Name: FE_PES...,0.000000,46.000000,3797.126973,3597.877847,0.000000,30.000000


In [166]:
cyclist_profile_options = (
    cyclist_profile[[
    'SEXO',
    'ZONA',
    'PE_BICI',
    'TIPO_DOM',
    'GRAU_INS',
    'RAÇA',
    'MOT_SRES',
    'FE_PESS'
    ]]
)

def get_value_counts(group):
    results = {}
    total_weight = group['FE_PESS'].sum()
    for col in cyclist_profile_options.columns:
        weighted_counts = group.groupby(col)['FE_PESS'].sum()
        results[col] = weighted_counts / total_weight
    return pd.Series(results)

cyclist_profile_options = cyclist_profile_options.groupby('ZONA').apply(get_value_counts)
cyclist_profile_options = cyclist_profile_options.drop(columns=['FE_PESS', 'ZONA'])

cyclist_profile_numerics = cyclist_profile[list(set(cyclist_profile.columns)-set(cyclist_profile_options.columns))]
cyclist_profile_numerics = cyclist_profile_numerics.groupby('ZONA').apply(lambda x: x.apply(lambda y: (y * x['FE_PESS']).sum() / x['FE_PESS'].sum() if y.name != 'FE_PESS' else y.sum()))
cyclist_profile_numerics = cyclist_profile_numerics.drop(columns=['FE_PESS'])

cyclist_profile = cyclist_profile_options.merge(cyclist_profile_numerics, on='ZONA', how='inner')

C:\Users\João Rahal\AppData\Local\Temp\ipykernel_14208\1684131956.py:22: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cyclist_profile_options = cyclist_profile_options.groupby('ZONA').apply(get_value_counts)


Agora que temos as zonas devidamente separadas, é uma questão de dispor isso nos mapas folium, para você poder analisar individualmente cada zona. Por enquanto eu vou fazer a geral, pois a amostragem é muito pequena, mas assim que conseguir consertar os microdados, mudarei para o censo.

### Análises:

1 - Sexo, idade, renda, duração, pe_bici, tipo-dom e grau-ins pra cada zona

2 - Geral qt_auto, qt_moto, mot_sres

3 - Apresentar a falta de microdados do censo

4 - Apresentar o PNB score atual e o com as rotas teorizadas

5 - Apresentar o PNB score por região

In [ ]:
1 

### Limpando o monstro que é o arquivo de pessoas (diminuí para 5% do tamanho original)

In [11]:
import pandas as pd

def read_ibge_microdata(txt_path, layout_path, sheet_name='PESS'):
    # Read layout sheet
    layout = pd.read_excel(layout_path, sheet_name=sheet_name, engine='odf').iloc[1:]
    
    # Extract variable names and positions
    var_names = layout.iloc[:, 0].astype(str).tolist()
    starts = layout.iloc[:, 7].astype(float).astype(int).tolist()
    ends = layout.iloc[:, 8].astype(float).astype(int).tolist()
    
    # Build colspecs
    colspecs = [(start - 1, end) for start, end in zip(starts, ends)]
    
    # Read fixed-width file
    df = pd.read_fwf(txt_path, colspecs=colspecs, names=var_names)
    
    return df

txt_path = '.\\data\\Amostra_Pessoas_35_RMSP.txt'
layout_path = '.\\Layout_microdados_Amostra.ods'

df = read_ibge_microdata(txt_path, layout_path)

dict_code_prof = {
    'V0010': 'peso_amostral',
    'V0011': 'setor',
    'V0601':{'sexo':{'1':'Masculino','2':'Feminino'}},
    'V0606':{'cor':{'1':'Branca','2':'Preta','3':'Amarela','4':'Parda','5':'Indigena','9':'Sem_declaracao'}},
    'V6400':{'nivel_de_instrucao':{'1':'Sem_instrucao','2':'fundamental_completo','3':'medio_completo','4':'superior_completo'}},
    'V0640':{'estado_civil':{'1':'casado','5':'solteiro','4':'viuvo','3':'divorciado','2':'desquitado'}},
    'V0645':{'trabalho':{'1':'um','2':'dois_ou_mais','':'nenhum'}},
    'V6526':'renda_em_salarios_minimos',
    'V0653':'horas_trabalhadas',
    'V0662':{'tempo_de_deslocamento_ao_trabalho':{'1':'<=5min','2':'6-30min','3':'31min-1h','4':'1-2h','5':'>2h'}},
    'V6940':{'categoria_profissional':{'1':'empregado_clt','2':'empregado_estatuario(militares_inclusos)','3':'empregado_sem_clt','4':'conta_propria','5':'empregador','6':'nao_remunerado','7':'trabalhador_subsistente'}},
    'V5080':'rendimento_familiar_per_capita_em_salarios_minimos',
    'V1005':{'situacao_do_setor':{'1':'area_urbanizada','2':'area_nao_urbanizada','3':'area_urbanizada_isolada','4':'area_rural_de_extensao_urbana','5':'aglomerado_rural','6':'aglomerado_rural','7':'aglomerado_rural','8':'area_rural_exclusive_aglomerado_rural'}},
    'V0221':{'motocicleta_para_uso_particular':{'1':'possui','2':'nao_possui'}},
    'V0222':{'automovel_para_uso_particular':{'1':'possui','2':'nao_possui'}},
}

unused_cols = [col for col in df.columns if col not in dict_code_prof.keys()]
df = df.drop(columns=unused_cols)

rename_map = {}

for key, value in dict_code_prof.items():
    if type(value) == str:
        rename_map[key] = value
    elif type(value) == dict:
        rename_map[key] = list(value.keys())[0]

df = df.rename(columns=rename_map)

cut_df = df[[col for col in df.columns if col in dict_code_prof.keys()]]
cut_df = df[(df['setor']>=3550308000000) & (df['setor']<=3550309000000)]		
cut_df.to_csv('biker_profile.csv')